In [1]:
from recbole.quick_start import load_data_and_model, run_recbole
import torch
import pandas as pd


import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import NeuMF
from recbole.model.sequential_recommender import SASRec
from recbole.trainer import Trainer
from recbole.utils import get_model, get_trainer, init_seed, init_logger


/home/mvarasteh/.conda/envs/popsteer/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-29 19:13:10,173	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-04-29 19:13:10,407	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
dataset_name='ml-1m'

In [ ]:
config = Config(
    model='SASRec',

    dataset='ml-1m',   
    config_file_list=['configs/SASRec_ml-1m.yaml'])


# Init seed and logger
init_seed(config['seed'], config['reproducibility'])
init_logger(config)

# Load dataset and map user/item IDs
dataset = create_dataset(config)

print(f"Num users: {dataset.user_num}, Num items: {dataset.item_num}")


# Split into train/valid/test and build user-item mappings
train_data, valid_data, test_data = data_preparation(config, dataset)

# Build model
model = SASRec(config, train_data.dataset).to(config['device'])

# Build trainer
trainer = Trainer(config, model)

# Train the model
best_valid_score, best_valid_result = trainer.fit(
    train_data,
    valid_data,
    saved=True,          # saves best checkpoint
    show_progress=True   # progress bar per epoch
)

print(f"Best validation score: {best_valid_score}")
print(f"Best validation result: {best_valid_result}")



## Extracting embeddings

In [60]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict

# ── 0. LOAD MODEL AND DATASET ───────────────────────────────────────────────
from recbole.quick_start import load_data_and_model

config, sasrec, dataset, train_data, valid_data, test_data = load_data_and_model(
     model_file='saved/SASRec-Apr-29-2026_15-23-50.pth'
    #model_file='saved/SASRec-Apr-29-2026_15-23-50.pth'
)
sasrec.eval()
device = config['device']

# ── 1. LOAD RAW METADATA ────────────────────────────────────────────────────
# read the raw file and inspect first

movies = pd.read_csv(
    'dataset/ml-1m/ml-1m.item',
    sep='\t',
    engine='python'
)

# rename to simple names
movies = movies.rename(columns={
    'item_id':        'item_id',
    'movie_title': 'title',
    'release_year':   'year',
    'genre':      'genre'
})

# genres are space-separated in this file (not pipe-separated)
# e.g. "Animation Children's Comedy" instead of "Animation|Children|Comedy"
# so split on space when building genre pools


ratings = pd.read_csv(
    'dataset/ml-1m/ml-1m.inter',
    sep='\t',
    engine='python'
)


# rename to simple names
ratings = ratings.rename(columns={
    'user_id:token':    'user_id',
    'item_id:token':    'item_id',
    'rating:float':     'rating',
    'timestamp:float':  'timestamp'
})






n_items = dataset.item_num
n_users = dataset.user_num



# ── 2. DEFINE CONCEPT ITEM POOLS ────────────────────────────────────────────
# split each genre string by space, flatten, and deduplicate
SEP = '|'

all_unique_genres = sorted(set(
    g.strip()
    for genres in movies['genres'].dropna()
    for g in str(genres).split(SEP)
    if g.strip()
))


# genre pools — items belonging to each genre
genre_pools = defaultdict(list)
for iid in range(n_items):
    row=movies[movies['item_id']==iid]
    #iid = int(row['item_id'])
    for g in str(row['genres']).split('|'):   
        g = g.strip()
        if g in all_unique_genres:                 
            genre_pools[g].append(iid)



29 Apr 22:21    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 42
state = INFO
reproducibility = True
data_path = dataset/ml-1m
checkpoint_dir = saved
show_progress = False
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 100
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = True
metrics = ['Recall', 'NDCG', 'Precision', 'Hit']
topk = [10, 20]
valid_metric = NDCG@10
valid_metric_bigger = True
eval_batch_size = 4096
metric_decimal_place = 4

Dataset Hyper Parameters:
f

In [4]:
# popularity pool — top 10% most interacted items
inter_df = pd.DataFrame({
    'user_id': dataset.inter_feat['user_id'].numpy(),
    'item_id': dataset.inter_feat['item_id'].numpy(),
})
item_counts = inter_df.groupby('item_id')['user_id'].count()
pop_map = {}
for raw_id, count in item_counts.items():
    #iid = item_id_map.get(str(raw_id))
    iid=raw_id
    if iid is not None:
        pop_map[int(iid)] = count

counts_arr = np.array([pop_map.get(i, 0) for i in range(n_items)])
pop_threshold = np.percentile(counts_arr, 90)
popularity_pool = [i for i in range(n_items) if counts_arr[i] >= pop_threshold]

In [5]:
# niche pool — bottom 10% least interacted items
niche_threshold = np.percentile(counts_arr, 10)
niche_pool = [i for i in range(n_items) if 0 < counts_arr[i] <= niche_threshold]


In [6]:
# era pools
movies['year'] = pd.to_numeric(movies['released_year'], errors='coerce')

classic_pool   = [int(r['item_id']) for _, r in movies.iterrows() if r['year'] < 1970]
retro_pool     = [int(r['item_id']) for _, r in movies.iterrows() if 1970 <= r['year'] < 1990]
modern_pool    = [int(r['item_id']) for _, r in movies.iterrows() if 1990 <= r['year'] < 2000]
contemporary_pool = [int(r['item_id']) for _, r in movies.iterrows() if r['year'] >= 2000]



## Combining concepts

In [7]:
concept_pools = {g: genre_pools[g] for g in all_unique_genres}
concept_pools['popularity']   = popularity_pool
concept_pools['niche']        = niche_pool


concept_pools['classic']      = classic_pool
concept_pools['retro']        = retro_pool
concept_pools['modern']       = modern_pool
concept_pools['contemporary'] = contemporary_pool

## CONCEPT PROTOTYPE VECTORS 

In [57]:

# ── 3. SYNTHETIC USER FORWARD PASS (SASRec) ─────────────────────────────────
N_SYNTHETIC = 1000
K_ITEMS     = 50  # matches SASRec's max_seq_length

def get_synthetic_user_repr_sasrec(item_pool, n_synthetic, k_items, model, device):
    """
    For each synthetic user:
      1. Sample a sequence of K items from the concept pool
      2. Left-align in a max_seq_len tensor (pad with 0 on the right)
      3. Run through SASRec → get 128-dim final hidden state
    Returns: tensor [n_synthetic, hidden_size]
    """
    user_reprs = []
    item_pool_tensor = torch.tensor(item_pool, dtype=torch.long)
    max_seq_len = model.max_seq_length  # 50

    model.eval()
    with torch.no_grad():
        for _ in range(n_synthetic):
            # sample K items as the user's interaction sequence
            sampled_idx = torch.randint(0, len(item_pool), (k_items,))
            sequence    = item_pool_tensor[sampled_idx]

            seq_len = min(k_items, max_seq_len)

            # left-align: items at positions 0..seq_len-1, pad with 0 on the right
            padded = torch.zeros(max_seq_len, dtype=torch.long)
            padded[:seq_len] = sequence[:seq_len]

            seq_tensor     = padded.unsqueeze(0).to(device)                        # [1, 50]
            seq_len_tensor = torch.tensor([seq_len], dtype=torch.long).to(device)  # [1]

            # SASRec forward returns [1, hidden_size] directly
            seq_output = model.forward(seq_tensor, seq_len_tensor)  # [1, 128]

            user_reprs.append(seq_output.squeeze(0).cpu())

    return torch.stack(user_reprs, dim=0)  # [n_synthetic, 128]


# ── 4. COMPUTE CONCEPT PROTOTYPE VECTORS ────────────────────────────────────
concept_prototypes  = {}
concept_all_vectors = {}

for concept, pool in concept_pools.items():
    if len(pool) < K_ITEMS:
        print(f"Skipping {concept} — pool too small ({len(pool)} items)")
        continue

    # exclude padding ID just in case
    pool = [i for i in pool if i != 0]

    print(f"Processing concept: {concept} ...")
    vecs      = get_synthetic_user_repr_sasrec(pool, N_SYNTHETIC, K_ITEMS, sasrec, device)
    prototype = vecs.mean(dim=0)

    concept_all_vectors[concept] = vecs
    concept_prototypes[concept]  = prototype
    print(f"  {concept:20s}: vectors {vecs.shape}, prototype {prototype.shape}")

# ── 5. SAVE EVERYTHING ──────────────────────────────────────────────────────
torch.save({
    'concept_prototypes':  concept_prototypes,
    'concept_all_vectors': concept_all_vectors,
    'concept_names':       list(concept_prototypes.keys()),
    'n_synthetic':         N_SYNTHETIC,
    'k_items':             K_ITEMS,
    'hidden_size':         sasrec.hidden_size,
    'model_type':          'SASRec',
}, f'concept_activations_sasrec_{dataset_name}.pt')

print("\nSaved to concept_activations_sasrec.pt")

# ── 6. SANITY CHECK ─────────────────────────────────────────────────────────
print("\nPrototype norms:")
for concept, proto in concept_prototypes.items():
    print(f"  {concept:20s}: norm = {proto.norm():.4f}")

print("\nCosine similarity between selected concepts:")
concepts_to_compare = ['Drama', 'Action', 'Romance', 'Horror', 'popularity', 'niche']
concepts_to_compare = [c for c in concepts_to_compare if c in concept_prototypes]

for i, c1 in enumerate(concepts_to_compare):
    for c2 in concepts_to_compare[i+1:]:
        v1  = concept_prototypes[c1]
        v2  = concept_prototypes[c2]
        sim = torch.nn.functional.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0))
        print(f"  {c1:15s} vs {c2:15s}: {sim.item():.3f}")

# ── 7. DIFFERENCE CHECK ─────────────────────────────────────────────────────
# verify prototypes actually differ across concepts
print("\nL2 distance between concepts (should be non-zero):")
for c1 in ['Drama', 'Action']:
    for c2 in ['Romance', 'Horror', 'Animation']:
        if c1 in concept_prototypes and c2 in concept_prototypes:
            dist = (concept_prototypes[c1] - concept_prototypes[c2]).norm().item()
            print(f"  {c1:10s} vs {c2:10s}: {dist:.4f}")

Skipping Action — pool too small (0 items)
Processing concept: Adventure ...
  Adventure           : vectors torch.Size([1000, 128]), prototype torch.Size([128])
Skipping Animation — pool too small (14 items)
Processing concept: Children's ...
  Children's          : vectors torch.Size([1000, 128]), prototype torch.Size([128])
Processing concept: Comedy ...
  Comedy              : vectors torch.Size([1000, 128]), prototype torch.Size([128])
Skipping Crime — pool too small (45 items)
Skipping Documentary — pool too small (0 items)
Processing concept: Drama ...
  Drama               : vectors torch.Size([1000, 128]), prototype torch.Size([128])
Skipping Fantasy — pool too small (21 items)
Skipping Film-Noir — pool too small (10 items)
Skipping Horror — pool too small (24 items)
Skipping Musical — pool too small (15 items)
Skipping Mystery — pool too small (31 items)
Processing concept: Romance ...
  Romance             : vectors torch.Size([1000, 128]), prototype torch.Size([128])
Proces

In [58]:
# Run this ONCE before anything else in your notebook
import recbole.evaluator.metrics as recbole_metrics
import numpy as np

# Monkey-patch np.float back for this session
if not hasattr(np, 'float'):
    np.float = float

print("Patched np.float ✓")

Patched np.float ✓


In [59]:
import numpy as np

# ── Compatibility patch for RecBole + NumPy >= 1.24 ──
if not hasattr(np, 'float'):   np.float   = float
if not hasattr(np, 'int'):     np.int     = int
if not hasattr(np, 'complex'): np.complex = complex
if not hasattr(np, 'bool'):    np.bool    = bool
if not hasattr(np, 'object'):  np.object  = object
if not hasattr(np, 'str'):     np.str     = str

In [66]:
import os
import traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from recbole.quick_start import load_data_and_model
from recbole.trainer import Trainer
from recbole.model.abstract_recommender import SequentialRecommender


# ── 0. LOAD FROZEN SASRec ───────────────────────────────────────────────────
original_load = torch.load
def patched_load(f, **kwargs):
    kwargs['weights_only'] = False
    return original_load(f, **kwargs)
torch.load = patched_load


torch.load = original_load

device = config['device']
sasrec.eval()
for p in sasrec.parameters():
    p.requires_grad = False

n_items     = dataset.item_num
hidden_size = sasrec.hidden_size
max_seq_len = sasrec.max_seq_length

print(f"Device:      {device}")
print(f"n_items:     {n_items}")
print(f"hidden_size: {hidden_size}")


# ── 1. LOAD FROZEN CONCEPT PROTOTYPES ───────────────────────────────────────
saved = torch.load('concept_activations_sasrec.pt', weights_only=False)
concept_prototypes = saved['concept_prototypes']

concept_names      = saved['concept_names']

n_concepts         = len(concept_names)

concept_matrix = torch.stack(
    [concept_prototypes[c] for c in concept_names], dim=0
).to(device)
concept_matrix.requires_grad = False

print(f"\nLoaded {n_concepts} concept prototypes")
print(f"Concept matrix shape: {concept_matrix.shape}")
print(f"Concepts: {concept_names}")


# ── REMOVE the torch.load patch entirely ────────────────────────────────────

# ── 2. CONCEPT BOTTLENECK MODULE (fixed) ─────────────────────────────────────
class ConceptBottleneck(nn.Module):
    def __init__(self, concept_matrix, hidden_size=128,
                 bottleneck_size=64, residual_weight=0.1):
        super().__init__()
        self.register_buffer('concepts', concept_matrix)
        self.n_concepts  = concept_matrix.shape[0]
        self.hidden_size = hidden_size

        # FIXED residual weight — not learnable
        self.residual_weight = residual_weight  # just a float, not nn.Parameter

        self.decoder = nn.Sequential(
            nn.Linear(self.n_concepts, bottleneck_size),
            nn.ReLU(),
            nn.Linear(bottleneck_size, hidden_size),
        )

    def forward(self, h):
        h_norm      = F.normalize(h, dim=-1)
        c_norm      = F.normalize(self.concepts, dim=-1)
        concept_act = h_norm @ c_norm.T
        h_concept   = self.decoder(concept_act)
        h_prime     = h_concept + self.residual_weight * h  # fixed weight
        return h_prime, concept_act


# ── 3. CBM MODEL (fixed calculate_loss) ──────────────────────────────────────
class SASRecCBM(SequentialRecommender):
    def __init__(self, config, dataset, sasrec, bottleneck):
        super().__init__(config, dataset)
        self.sasrec     = sasrec
        self.bottleneck = bottleneck
        for p in self.sasrec.parameters():
            p.requires_grad = False

    def _encode(self, interaction):
        item_seq     = interaction[self.ITEM_SEQ]
        item_seq_len = interaction[self.ITEM_SEQ_LEN]
        with torch.no_grad():
            h = self.sasrec.forward(item_seq, item_seq_len)
        h_prime, concept_act = self.bottleneck(h)
        return h_prime, concept_act

    def calculate_loss(self, interaction):
        h_prime, _  = self._encode(interaction)
        pos_items   = interaction[self.POS_ITEM_ID]
        item_embs   = self.sasrec.item_embedding.weight
        logits      = h_prime @ item_embs.T
        logits[:, 0] = float('-inf')   # ← mask padding token
        return F.cross_entropy(logits, pos_items)

    def full_sort_predict(self, interaction):
        h_prime, _ = self._encode(interaction)
        return h_prime @ self.sasrec.item_embedding.weight.T

    def predict(self, interaction):
        h_prime, _  = self._encode(interaction)
        test_item   = interaction[self.POS_ITEM_ID]
        item_embs   = self.sasrec.item_embedding(test_item)
        return (h_prime * item_embs).sum(dim=-1)







# ── 4. BASELINE WRAPPER ──────────────────────────────────────────────────────
class OriginalSASRecWrapper(SequentialRecommender):
    """RecBole-compatible wrapper for frozen SASRec — eval only."""
    def __init__(self, config, dataset, sasrec):
        super().__init__(config, dataset)
        self.sasrec = sasrec
        for p in self.sasrec.parameters():
            p.requires_grad = False

    def calculate_loss(self, interaction):
        return torch.tensor(0.0, device=device)

    def full_sort_predict(self, interaction):
        item_seq     = interaction[self.ITEM_SEQ]
        item_seq_len = interaction[self.ITEM_SEQ_LEN]
        with torch.no_grad():
            h = self.sasrec.forward(item_seq, item_seq_len)
        return h @ self.sasrec.item_embedding.weight.T

    def predict(self, interaction):
        item_seq     = interaction[self.ITEM_SEQ]
        item_seq_len = interaction[self.ITEM_SEQ_LEN]
        test_item    = interaction[self.POS_ITEM_ID]
        with torch.no_grad():
            h = self.sasrec.forward(item_seq, item_seq_len)
        return (h * self.sasrec.item_embedding(test_item)).sum(dim=-1)


# ── 5. INSTANTIATE ───────────────────────────────────────────────────────────
bottleneck = ConceptBottleneck(
    concept_matrix,
    hidden_size=hidden_size,
    bottleneck_size=64,
    residual_weight=0.1,   # fixed at 0.1 — not learnable
).to(device)

cbm_model = SASRecCBM(config, dataset, sasrec, bottleneck).to(device)

trainable = sum(p.numel() for p in cbm_model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {trainable:,}")
print(f"Trainable param names:")
for name, p in cbm_model.named_parameters():
    if p.requires_grad:
        print(f"  {name}: {p.shape}")


# ── 6. CONFIG (fixed casing) ─────────────────────────────────────────────────
config['metrics']      = ['recall', 'ndcg', 'hit']   # ← lowercase
config['topk']         = [10]
config['valid_metric'] = 'ndcg@10'                    # ← lowercase
config['eval_args'] = {
    'split':    {'LS': 'valid_and_test'},
    'order':    'TO',
    'group_by': 'user',
    'mode':     {'valid': 'full', 'test': 'full'},
}
config['epochs']         = 30
config['learning_rate']  = 1e-3
config['eval_step']      = 1
config['stopping_step']  = 10
config['checkpoint_dir'] = 'saved'


# ── 7. TRAIN ─────────────────────────────────────────────────────────────────
trainer = Trainer(config, cbm_model)

print("\n" + "="*60)
print("TRAINING")
print("="*60)

try:
    best_valid_score, best_valid_result = trainer.fit(
        train_data,
        valid_data,
        saved=True,
        show_progress=True,
    )
    print(f"\nBest validation score : {best_valid_score:.4f}")
    print(f"Best validation result: {best_valid_result}")
except Exception as e:
    print(f"\nERROR during training: {e}")
    traceback.print_exc()
    raise


# ── 8. EVALUATE CBM ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATION — CBM")
print("="*60)

try:
    test_result = trainer.evaluate(
        test_data, load_best_model=False, show_progress=True
    )
    print("\nCBM Test metrics:")
    for m, v in test_result.items():
        print(f"  {m:15s}: {v:.4f}")
except Exception as e:
    print(f"\nERROR during CBM evaluation: {e}")
    traceback.print_exc()
    raise


# ── 9. EVALUATE BASELINE ─────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATION — Original SASRec baseline")
print("="*60)

try:
    baseline         = OriginalSASRecWrapper(config, dataset, sasrec).to(device)
    baseline_trainer = Trainer(config, baseline)

    baseline_result = baseline_trainer.evaluate(
        test_data, load_best_model=False, show_progress=True
    )
    print("\nBaseline Test metrics:")
    for m, v in baseline_result.items():
        print(f"  {m:15s}: {v:.4f}")
except Exception as e:
    print(f"\nERROR during baseline evaluation: {e}")
    traceback.print_exc()
    raise


# ── 10. INTERPRETABILITY TAX ─────────────────────────────────────────────────
print("\n" + "="*60)
print("INTERPRETABILITY TAX (CBM vs original SASRec)")
print("="*60)

for m in ['ndcg@10', 'recall@10', 'hit@10']:
    base_val = baseline_result.get(m, 0)
    cbm_val  = test_result.get(m, 0)
    drop     = (base_val - cbm_val) / (base_val + 1e-8) * 100
    print(f"  {m}: {base_val:.4f} → {cbm_val:.4f}  ({drop:+.1f}% relative)")


# ── 11. SAVE ──────────────────────────────────────────────────────────────────
torch.save({
    'bottleneck_state': cbm_model.bottleneck.state_dict(),
    'concept_names':    concept_names,
    'val_metrics':      best_valid_result,
    'test_metrics':     test_result,
    'baseline_metrics': baseline_result,
    'config': {
        'n_concepts':      n_concepts,
        'hidden_size':     hidden_size,
        'residual_weight': 0.3,
        'lr':              config['learning_rate'],
        'epochs':          config['epochs'],
    }
}, 'cbm_sasrec.pt')

print("\nSaved to cbm_sasrec.pt")


# ── 12. EXAMPLE EXPLANATION ───────────────────────────────────────────────────
print("\n" + "="*60)
print("EXAMPLE EXPLANATION")
print("="*60)

from recbole.data.interaction import Interaction

cbm_model.eval()
with torch.no_grad():
    try:
        batch = next(iter(test_data))
        if isinstance(batch, tuple):
            batch = batch[0]
        batch = batch.to(device)

        single_inter = Interaction({
            k: v[:1] for k, v in batch.interaction.items()
        })

        h_prime, concept_act = cbm_model._encode(single_inter)
        scores = h_prime @ sasrec.item_embedding.weight.T     # [1, n_items]
        scores[:, 0] = float('-inf')                          # mask padding

        top5_items   = scores[0].topk(5).indices.cpu().tolist()
        top_concepts = concept_act[0].topk(min(5, n_concepts))
        bot_concepts = (-concept_act[0]).topk(min(3, n_concepts))

        print(f"\nTop 5 recommended item IDs: {top5_items}")
        print("\nTop active concepts:")
        for v, idx in zip(top_concepts.values, top_concepts.indices):
            print(f"  {concept_names[idx]:20s}  activation: {v.item():.4f}")
        print("\nMost suppressed concepts:")
        for v, idx in zip(bot_concepts.values, bot_concepts.indices):
            print(f"  {concept_names[idx]:20s}  activation: {-v.item():.4f}")

    except Exception as e:
        print(f"ERROR during example explanation: {e}")
        traceback.print_exc()

Device:      cuda
n_items:     3707
hidden_size: 128

Loaded 22 concept prototypes
Concept matrix shape: torch.Size([22, 128])
Concepts: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'popularity', 'niche', 'classic', 'retro', 'modern', 'contemporary']

Trainable parameters: 9,792
Trainable param names:
  bottleneck.decoder.0.weight: torch.Size([64, 22])
  bottleneck.decoder.0.bias: torch.Size([64])
  bottleneck.decoder.2.weight: torch.Size([128, 64])
  bottleneck.decoder.2.bias: torch.Size([128])

TRAINING


Train     0:   0%|                                                          | 0/480 [00:00<?, ?it/s]/home/mvarasteh/post-hoc/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
Train     0:   0%|                                 | 0/480 [00:00<?, ?it/s, GPU RAM: 2.79 G/15.72 G]:   0%|                                 | 0/480 [00:00<?, ?it/s, GPU RAM: 2.79 G/15.72 G]:   0%|                                 | 0/480 [00:00<?, ?it/s, GPU RAM: 2.79 G/15.72 G]:   1%|▏                        | 3/480 [00:00<00:20, 23.11it/s, GPU RAM: 2.79 G/15.72 G]:   1%|▏                        | 3/480 [00:00<00:20, 23.11it/s, GPU RAM: 2.79 G/15.72 G]:   1%|▏                        | 3/480 [00:00<00:20, 23.11it/s, GPU RAM: 2.79 G/15.72 G]:   1%|▏                        | 3/480 [00:00<00:20, 23.11it/s, GPU RAM: 2.79 G/15.72 G]:   1%|▎                    


Best validation score : 0.1275
Best validation result: OrderedDict([('recall@10', 0.2364), ('ndcg@10', 0.1275), ('hit@10', 0.2364)])

EVALUATION — CBM


Evaluate   :   0%|                                                            | 0/2 [00:00<?, ?it/s]:   0%|                                   | 0/2 [00:00<?, ?it/s, GPU RAM: 2.79 G/15.72 G]:   0%|                                   | 0/2 [00:00<?, ?it/s, GPU RAM: 2.79 G/15.72 G]: 100%|███████████████████████████| 2/2 [00:00<00:00, 22.39it/s, GPU RAM: 2.79 G/15.72 G]



CBM Test metrics:
  recall@10      : 0.2214
  ndcg@10        : 0.1150
  hit@10         : 0.2214

EVALUATION — Original SASRec baseline


Evaluate   :   0%|                                                            | 0/2 [00:00<?, ?it/s]:   0%|                                   | 0/2 [00:00<?, ?it/s, GPU RAM: 2.79 G/15.72 G]:   0%|                                   | 0/2 [00:00<?, ?it/s, GPU RAM: 2.79 G/15.72 G]: 100%|███████████████████████████| 2/2 [00:00<00:00, 22.37it/s, GPU RAM: 2.79 G/15.72 G]



Baseline Test metrics:
  recall@10      : 0.2717
  ndcg@10        : 0.1476
  hit@10         : 0.2717

INTERPRETABILITY TAX (CBM vs original SASRec)
  ndcg@10: 0.1476 → 0.1150  (+22.1% relative)
  recall@10: 0.2717 → 0.2214  (+18.5% relative)
  hit@10: 0.2717 → 0.2214  (+18.5% relative)

Saved to cbm_sasrec.pt

EXAMPLE EXPLANATION

Top 5 recommended item IDs: [140, 212, 141, 377, 211]

Top active concepts:
  retro                 activation: 0.2239
  Thriller              activation: 0.1298
  Mystery               activation: 0.1247
  contemporary          activation: 0.1154
  classic               activation: 0.0865

Most suppressed concepts:
  modern                activation: -0.1670
  Sci-Fi                activation: -0.0982
  Action                activation: -0.0776


In [69]:
# ── 13. SAVE CONCEPT ACTIVATIONS FOR ALL USERS ───────────────────────────────
print("\n" + "="*60)
print("COLLECTING CONCEPT ACTIVATIONS FOR ALL USERS")
print("="*60)

cbm_model.eval()
all_concept_acts = []
all_user_ids     = []
all_item_seqs    = []

with torch.no_grad():
    for batch in test_data:
        if isinstance(batch, tuple):
            batch = batch[0]
        batch = batch.to(device)

        # collect user ids and sequences
        user_ids = batch['user_id'].cpu().numpy()          # [B]
        item_seq = batch[cbm_model.ITEM_SEQ].cpu().numpy() # [B, seq_len]

        # get concept activations
        _, concept_act = cbm_model._encode(batch)          # [B, N]

        all_concept_acts.append(concept_act.cpu())
        all_user_ids.append(torch.tensor(user_ids))
        all_item_seqs.append(torch.tensor(item_seq))

# stack everything
all_concept_acts = torch.cat(all_concept_acts, dim=0)  # [n_users, N_concepts]
all_user_ids     = torch.cat(all_user_ids,     dim=0)  # [n_users]
all_item_seqs    = torch.cat(all_item_seqs,    dim=0)  # [n_users, seq_len]

print(f"Total users:        {all_user_ids.shape[0]}")
print(f"Concept activations shape: {all_concept_acts.shape}")

# ── Save as .pt ───────────────────────────────────────────────────────────────
torch.save({
    'user_ids':     all_user_ids,           # [n_users]
    'concept_acts': all_concept_acts,       # [n_users, N_concepts]
    'item_seqs':    all_item_seqs,          # [n_users, seq_len]
    'concept_names': concept_names,         # list of N concept names
}, 'user_concept_activations.pt')

print("Saved to user_concept_activations.pt")

# ── Save as CSV (human-readable) ──────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(
    all_concept_acts.numpy(),
    columns=concept_names
)
df.insert(0, 'user_id', all_user_ids.numpy())
df.to_csv('user_concept_activations.csv', index=False)
print("Saved to user_concept_activations.csv")

# ── Quick stats ───────────────────────────────────────────────────────────────
print("\nPer-concept activation stats:")
print(f"{'Concept':20s}  {'Mean':>8s}  {'Std':>8s}  {'Min':>8s}  {'Max':>8s}")
print("-" * 60)
for i, name in enumerate(concept_names):
    vals = all_concept_acts[:, i]
    print(f"  {name:18s}  {vals.mean():8.4f}  {vals.std():8.4f}  "
          f"{vals.min():8.4f}  {vals.max():8.4f}")


COLLECTING CONCEPT ACTIVATIONS FOR ALL USERS
Total users:        6040
Concept activations shape: torch.Size([6040, 22])
Saved to user_concept_activations.pt
Saved to user_concept_activations.csv

Per-concept activation stats:
Concept                   Mean       Std       Min       Max
------------------------------------------------------------
  Action                0.0075    0.0681   -0.2122    0.2071
  Adventure            -0.0001    0.0791   -0.3548    0.2515
  Animation             0.0078    0.0801   -0.2835    0.2341
  Comedy               -0.0034    0.0825   -0.3178    0.2495
  Crime                 0.0128    0.0903   -0.2846    0.3697
  Documentary           0.0014    0.0625   -0.2376    0.2178
  Drama                -0.0284    0.0673   -0.2488    0.1921
  Fantasy               0.0179    0.0720   -0.2623    0.2238
  Horror                0.0023    0.0844   -0.2596    0.3339
  Musical              -0.0152    0.0759   -0.2388    0.2325
  Mystery               0.0729    0.0862 

In [70]:
# ── EXPLAIN INDIVIDUAL RECOMMENDED ITEMS FOR A USER ──────────────────────────

target_user = 41

# ── 1. Get user's hidden state and concept activations ───────────────────────
cbm_model.eval()
with torch.no_grad():

    # find user in test_data
    user_h        = None
    user_concepts = None

    for batch in test_data:
        if isinstance(batch, tuple):
            batch = batch[0]
        batch = batch.to(device)

        user_ids_batch = batch['user_id']
        mask = user_ids_batch == target_user
        if mask.any():
            # extract this user's row
            single = {k: v[mask][:1] for k, v in batch.interaction.items()}
            from recbole.data.interaction import Interaction
            single_inter = Interaction(single)

            user_h, user_concepts = cbm_model._encode(single_inter)  # [1,128], [1,N]
            user_h        = user_h.squeeze(0)        # [128]
            user_concepts = user_concepts.squeeze(0) # [N]
            break

    if user_h is None:
        raise ValueError(f"User {target_user} not found in test_data")

    # ── 2. Get top-K recommendations ─────────────────────────────────────────
    K = 10
    item_embs = sasrec.item_embedding.weight          # [n_items, 128]
    scores    = user_h @ item_embs.T                  # [n_items]
    scores[0] = float('-inf')                         # mask padding
    topk_ids  = scores.topk(K).indices.cpu().tolist() # [K] internal item IDs

    # ── 3. Compute per-item concept activations ───────────────────────────────
    # For each recommended item, compute its concept profile
    # via its embedding similarity to concept prototypes
    concept_matrix_norm = F.normalize(concept_matrix, dim=-1)  # [N, 128]

    item_concept_acts = {}
    for iid in topk_ids:
        item_emb  = item_embs[iid]                             # [128]
        item_norm = F.normalize(item_emb, dim=-1).unsqueeze(0) # [1, 128]
        item_acts = (item_norm @ concept_matrix_norm.T).squeeze(0)  # [N]
        item_concept_acts[iid] = item_acts.cpu()

    # ── 4. Print explanations ─────────────────────────────────────────────────
    id2token_item = dataset.field2id_token[dataset.iid_field]

    # Load titles if available
    import pandas as pd
    try:
        item_df   = pd.read_csv('dataset//ml-1m.item', sep='\t')
        id_col    = item_df.columns[0]
        title_col = item_df.columns[1]
        id_to_title = dict(zip(item_df[id_col].astype(str), item_df[title_col]))
    except:
        id_to_title = {}

    print(f"\n{'='*70}")
    print(f"RECOMMENDATION EXPLANATIONS — User {target_user}")
    print(f"{'='*70}")
    print(f"\nUser top concepts: "
          f"Thriller({user_concepts[concept_names.index('Thriller')]:.3f}), "
          f"popularity({user_concepts[concept_names.index('popularity')]:.3f}), "
          f"Animation({user_concepts[concept_names.index('Animation')]:.3f})")

    for rank, iid in enumerate(topk_ids, 1):
        orig_id   = id2token_item[iid]
        title     = id_to_title.get(str(orig_id), f'Item {orig_id}')
        item_acts = item_concept_acts[iid]            # [N]
        score     = scores[iid].item()

        # alignment = dot product between user concept vec and item concept vec
        alignment_per_concept = user_concepts.cpu() * item_acts  # [N]

        # top 3 concepts driving this recommendation
        top3 = alignment_per_concept.topk(3)
        bot3 = alignment_per_concept.topk(3, largest=False)

        print(f"\n{'─'*70}")
        print(f"  Rank {rank:2d} | Score: {score:.4f}")
        print(f"  Title: {title}  (internal ID: {iid}, original: {orig_id})")
        print(f"\n  WHY recommended (aligned concepts):")
        for val, idx in zip(top3.values, top3.indices):
            uval = user_concepts[idx].item()
            ival = item_acts[idx].item()
            print(f"    ✓ {concept_names[idx]:15s}  "
                  f"user={uval:+.3f}  item={ival:+.3f}  alignment={val:.4f}")

        print(f"\n  WHY NOT others (misaligned concepts):")
        for val, idx in zip(bot3.values, bot3.indices):
            uval = user_concepts[idx].item()
            ival = item_acts[idx].item()
            print(f"    ✗ {concept_names[idx]:15s}  "
                  f"user={uval:+.3f}  item={ival:+.3f}  alignment={val:.4f}")

    # ── 5. Save explanation table as CSV ─────────────────────────────────────
    rows = []
    for rank, iid in enumerate(topk_ids, 1):
        orig_id   = id2token_item[iid]
        title     = id_to_title.get(str(orig_id), f'Item {orig_id}')
        item_acts = item_concept_acts[iid]
        row = {
            'rank':       rank,
            'item_id':    orig_id,
            'title':      title,
            'score':      scores[iid].item(),
        }
        # add per-concept columns: user activation, item activation, alignment
        for i, name in enumerate(concept_names):
            row[f'user_{name}']      = user_concepts[i].item()
            row[f'item_{name}']      = item_acts[i].item()
            row[f'align_{name}']     = (user_concepts[i] * item_acts[i]).item()
        rows.append(row)

    explanation_df = pd.DataFrame(rows)
    explanation_df.to_csv(f'explanations_user_{target_user}.csv', index=False)
    print(f"\n\nSaved full explanation table to explanations_user_{target_user}.csv")


RECOMMENDATION EXPLANATIONS — User 41

User top concepts: Thriller(-0.026), popularity(-0.060), Animation(0.087)

──────────────────────────────────────────────────────────────────────
  Rank  1 | Score: 3.8483
  Title: Item 2960  (internal ID: 306, original: 2960)

  WHY recommended (aligned concepts):
    ✓ modern           user=-0.149  item=-0.212  alignment=0.0316
    ✓ Drama            user=-0.093  item=-0.203  alignment=0.0188
    ✓ Animation        user=+0.087  item=+0.112  alignment=0.0097

  WHY NOT others (misaligned concepts):
    ✗ retro            user=+0.124  item=-0.054  alignment=-0.0067
    ✗ Mystery          user=+0.040  item=-0.119  alignment=-0.0048
    ✗ Musical          user=+0.023  item=-0.155  alignment=-0.0035

──────────────────────────────────────────────────────────────────────
  Rank  2 | Score: 3.8295
  Title: Item 2776  (internal ID: 219, original: 2776)

  WHY recommended (aligned concepts):
    ✓ modern           user=-0.149  item=-0.163  alignment=0.0

In [79]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from recbole.data.interaction import Interaction

# ── CONFIG ──────────────────────────────────────────────────────────────────
TARGET_USER_ID      = 345
TOPK                = 20
CONCEPT_TO_SUPPRESS = 'drama'
SUPPRESS_VALUE      = -1

# ── SETUP ───────────────────────────────────────────────────────────────────
cbm_model.eval()
sasrec.eval()
device = config['device']

# ── 1. FIND CONCEPT INDEX ────────────────────────────────────────────────────
pop_idx = None
for i, name in enumerate(concept_names):
    if CONCEPT_TO_SUPPRESS in name.lower():
        pop_idx = i
        break

if pop_idx is None:
    raise ValueError(
        f"Concept '{CONCEPT_TO_SUPPRESS}' not found.\n"
        f"Available: {concept_names}"
    )

print(f"Suppressing concept: '{concept_names[pop_idx]}' (index={pop_idx})")


# ── 2. FIND TARGET USER IN TEST DATA ─────────────────────────────────────────
target_batch = None

for batch in test_data:
    if isinstance(batch, tuple):
        batch = batch[0]

    users = batch['user_id'].cpu().numpy()
    mask  = users == TARGET_USER_ID

    if mask.sum() > 0:
        idx = int(np.where(mask)[0][0])
        target_batch = Interaction({
            k: v[idx:idx+1].to(device)
            for k, v in batch.interaction.items()
        })
        break

if target_batch is None:
    raise ValueError(f"user_id={TARGET_USER_ID} not found in test_data")

print(f"Found user {TARGET_USER_ID}")
print("Batch keys:", list(target_batch.interaction.keys()))


# ── 3. GET FIELD NAMES ───────────────────────────────────────────────────────
item_seq_field     = cbm_model.ITEM_SEQ     if hasattr(cbm_model, 'ITEM_SEQ')     else 'item_id_list'
item_seq_len_field = cbm_model.ITEM_SEQ_LEN if hasattr(cbm_model, 'ITEM_SEQ_LEN') else 'item_length'

item_seq     = target_batch[item_seq_field]
item_seq_len = target_batch[item_seq_len_field]

print(f"item_seq field:     '{item_seq_field}', shape: {item_seq.shape}")
print(f"item_seq_len field: '{item_seq_len_field}', value: {item_seq_len}")


# ── 4. ORIGINAL RECOMMENDATION ───────────────────────────────────────────────
with torch.no_grad():
    h_prime, concept_act = cbm_model._encode(target_batch)

    scores_before = h_prime @ sasrec.item_embedding.weight.T
    scores_before[:, 0] = float('-inf')

    top_before          = scores_before[0].topk(TOPK)
    top20_before        = top_before.indices.cpu().tolist()
    top20_before_scores = top_before.values.cpu().tolist()

print(f"\nOriginal concept activations:")
for i, (name, val) in enumerate(zip(concept_names, concept_act[0].cpu().tolist())):
    print(f"  [{i:2d}] {name:20s}: {val:.4f}")


# ── 5. STEERED RECOMMENDATION ────────────────────────────────────────────────
with torch.no_grad():
    h = sasrec.forward(item_seq, item_seq_len)

    h_norm = F.normalize(h, dim=-1)
    c_norm = F.normalize(cbm_model.bottleneck.concepts, dim=-1)
    concept_act_steered = h_norm @ c_norm.T

    print(f"\nConcept activation BEFORE suppression: "
          f"{concept_names[pop_idx]} = {concept_act_steered[0, pop_idx].item():.4f}")

    steered_act = concept_act_steered.clone()
    steered_act[:, pop_idx] = SUPPRESS_VALUE

    h_concept       = cbm_model.bottleneck.decoder(steered_act)
    h_prime_steered = h_concept + cbm_model.bottleneck.residual_weight * h

    scores_after = h_prime_steered @ sasrec.item_embedding.weight.T
    scores_after[:, 0] = float('-inf')

    top_after          = scores_after[0].topk(TOPK)
    top20_after        = top_after.indices.cpu().tolist()
    top20_after_scores = top_after.values.cpu().tolist()


# ── 6. GENRE FREQUENCY COMPARISON ────────────────────────────────────────────
def genre_counts(top_ids, movies_df):
    subset = movies_df[movies_df['item_id'].isin(top_ids)]
    genres = (
        subset['genres']
        .astype(str)
        .str.split('|')
        .explode()
        .str.strip()
    )
    return genres.value_counts()


before_counts = genre_counts(top20_before, movies)
after_counts  = genre_counts(top20_after,  movies)

genre_df = (
    pd.concat([before_counts, after_counts], axis=1, keys=['before', 'after'])
      .fillna(0)
      .astype(int)
)
genre_df['delta'] = genre_df['after'] - genre_df['before']
genre_df = genre_df.sort_values('before', ascending=False)


# ── 7. PRINT RESULTS ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"User {TARGET_USER_ID} | Suppressing '{concept_names[pop_idx]}'")
print(f"{'='*60}")

print(f"\nTop{TOPK} BEFORE steering:")
before_df = movies[movies['item_id'].isin(top20_before)][['item_id', 'title', 'genres']].copy()
print(before_df.to_string(index=False))

print(f"\nTop{TOPK} AFTER steering:")
after_df = movies[movies['item_id'].isin(top20_after)][['item_id', 'title', 'genres']].copy()
print(after_df.to_string(index=False))

print(f"\nGenre frequency shift (Top{TOPK}):")
print(genre_df.to_string())

overlap = set(top20_before) & set(top20_after)
print(f"\nOverlap between before/after: {len(overlap)}/{TOPK} items")
print(f"New items introduced by steering: {TOPK - len(overlap)}")


# ── 8. SAVE RESULTS ──────────────────────────────────────────────────────────
result_df = pd.DataFrame({
    'rank':           list(range(1, TOPK + 1)),
    'before_item_id': top20_before,
    'after_item_id':  top20_after,
})



print(f"\nSaved: user_{TARGET_USER_ID}_steering_results.csv")
print(f"Saved: user_{TARGET_USER_ID}_genre_shift.csv")

Suppressing concept: 'Drama' (index=6)
Found user 345
Batch keys: ['user_id', 'item_id', 'rating', 'timestamp', 'item_length', 'item_id_list', 'rating_list', 'timestamp_list']
item_seq field:     'item_id_list', shape: torch.Size([1, 50])
item_seq_len field: 'item_length', value: tensor([24], device='cuda:0')

Original concept activations:
  [ 0] Action              : -0.0004
  [ 1] Adventure           : -0.0310
  [ 2] Animation           : -0.1106
  [ 3] Comedy              : -0.0706
  [ 4] Crime               : -0.0964
  [ 5] Documentary         : -0.0970
  [ 6] Drama               : -0.0313
  [ 7] Fantasy             : -0.0205
  [ 8] Horror              : 0.0332
  [ 9] Musical             : -0.0909
  [10] Mystery             : 0.0336
  [11] Romance             : -0.0282
  [12] Sci-Fi              : 0.0820
  [13] Thriller            : 0.0217
  [14] War                 : -0.0401
  [15] Western             : -0.0676
  [16] popularity          : -0.1056
  [17] niche               : 0.07

In [147]:
# ── 13. NICHE CONCEPT STEERING (sanity check) ────────────────────────────────
from recbole.data.interaction import Interaction
import pandas as pd

TARGET_USER_ID = 120
TOPK           = 20
ALPHAS         = [0.0, 0.5, 1.0, 2.0, 3.0]

print("\nAvailable concepts:", concept_names)

# convert pools to sets for O(1) lookup
niche_set     = set(niche_pool)
popularity_set = set(popularity_pool)
print(f"# niche items: {len(niche_set)} | # popular items: {len(popularity_set)}")

# locate niche concept
niche_idx = next(
    (i for i, name in enumerate(concept_names) if 'niche' in name.lower()),
    None
)
if niche_idx is None:
    raise ValueError(f"No 'niche' concept in {concept_names}")
print(f"Steering concept: {concept_names[niche_idx]} (index={niche_idx})")

# find target user's batch
cbm_model.eval()
target_batch = None
for batch in test_data:
    if isinstance(batch, tuple):
        batch = batch[0]
    users = batch['user_id'].cpu().numpy()
    mask  = users == TARGET_USER_ID
    if mask.sum() > 0:
        idx = int(np.where(mask)[0][0])
        target_batch = Interaction({
            k: v[idx:idx+1].to(device)
            for k, v in batch.interaction.items()
        })
        break
if target_batch is None:
    raise ValueError(f"user_id={TARGET_USER_ID} not found in test_data")

# sweep ALPHA — upweight niche concept
rows = []
with torch.no_grad():
    item_seq     = target_batch[cbm_model.ITEM_SEQ]
    item_seq_len = target_batch[cbm_model.ITEM_SEQ_LEN]
    h            = sasrec.forward(item_seq, item_seq_len)

    h_norm      = F.normalize(h, dim=-1)
    c_norm      = F.normalize(cbm_model.bottleneck.concepts, dim=-1)
    concept_act = h_norm @ c_norm.T

    for alpha in ALPHAS:
        steered_act = concept_act.clone()
        steered_act[:, niche_idx] = (
            concept_act[:, niche_idx] + alpha
        ).clamp(-1.0, 1.0)

        h_concept       = cbm_model.bottleneck.decoder(steered_act)
        h_prime_steered = h_concept + cbm_model.bottleneck.residual_weight * h

        scores = h_prime_steered @ sasrec.item_embedding.weight.T
        scores[:, 0] = float('-inf')   # mask padding
        top_ids = scores[0].topk(TOPK).indices.cpu().tolist()

        n_niche   = sum(1 for i in top_ids if i in niche_set)
        n_popular = sum(1 for i in top_ids if i in popularity_set)
        n_mid     = TOPK - n_niche - n_popular

        rows.append({
            'alpha':       alpha,
            'niche':       n_niche,
            'popular':     n_popular,
            'mid':         n_mid,
            'niche_pct':   round(100 * n_niche / TOPK, 1),
            'popular_pct': round(100 * n_popular / TOPK, 1),
        })

sweep_df = pd.DataFrame(rows)
print(f"\nTop{TOPK} composition vs niche-upweight α (user {TARGET_USER_ID}):")
print(sweep_df.to_string(index=False))


Available concepts: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'popularity', 'niche', 'classic', 'retro', 'modern', 'contemporary']
# niche items: 215 | # popular items: 343
Steering concept: niche (index=17)

Top20 composition vs niche-upweight α (user 120):
 alpha  niche  popular  mid  niche_pct  popular_pct
   0.0      0        2   18        0.0         10.0
   0.5      0        4   16        0.0         20.0
   1.0      0        5   15        0.0         25.0
   2.0      0        5   15        0.0         25.0
   3.0      0        5   15        0.0         25.0


1231